# Denoise Preprocessing

`reference/lif_pipeline.py`의 scanner-line denoise 알고리즘을 그대로 사용해 TIFF Z-stack을 registration 전에 정리합니다.

핵심 알고리즘은 `denoise_scanner_lines_series()`입니다. 이 노트북은 데이터 발견, 저장, QC figure, metadata 기록만 담당합니다.

## 0. Setup

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import sys

PROJECT_DIR = Path.cwd()
MPLCONFIGDIR = PROJECT_DIR / ".matplotlib"
MPLCONFIGDIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile

DATA_DIR = PROJECT_DIR / "data"
REFERENCE_DIR = PROJECT_DIR / "reference"
if str(REFERENCE_DIR) not in sys.path:
    sys.path.insert(0, str(REFERENCE_DIR))

from lif_pipeline import (
    SeriesData,
    _summarize_denoise_diag,
    _write_imagej_tiff,
    denoise_scanner_lines_series,
    discover_tiff_series,
    ensure_uint16,
    load_series_from_tiff_dir,
    sanitize_name,
)

CHANNEL_LABELS = ("DAPI", "Reference", "Target")

DENOISE_DETECTION_NEIGHBORHOOD = 31
DENOISE_DETECTION_THRESHOLD = 4.0
DENOISE_MIN_PUNCTUM_SIZE = 5

RUN_DENOISE = True
SAVE_DENOISED_STACKS = True
SAVE_QC_FIGURES = True
SAVE_METADATA_JSON = True

print({
    "project_dir": str(PROJECT_DIR),
    "data_dir": str(DATA_DIR),
    "reference_module": str(REFERENCE_DIR / "lif_pipeline.py"),
    "algorithm": "lif_pipeline.denoise_scanner_lines_series",
})

## 1. Input / Output Contract

In [ ]:
INPUTS = {
    "raw_stacks_current": "data/<lif_name>/<series_name>/stacks/<series_name>_stack.tif",
    "raw_stacks_legacy": "data/<lif_name>/<series_name>/stacks/{DAPI,Reference,Target}_stack.tif",
    "metadata": "data/<lif_name>/<series_name>/metadata.json",
}
OUTPUTS = {
    "denoised_stack": "data/<lif_name>/<series_name>/denoised_stacks/<series_name>_stack_denoised.tif",
    "preprocessing_metadata": "data/<lif_name>/<series_name>/preprocessing_metadata.json",
    "denoise_qc": "data/<lif_name>/<series_name>/denoise_qc/<series_name>_denoise_validation.png",
}

print("Inputs:")
for key, value in INPUTS.items():
    print(f"  {key}: {value}")
print("Outputs:")
for key, value in OUTPUTS.items():
    print(f"  {key}: {value}")

## 2. Series Discovery

In [ ]:
def discover_series(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    rows = []
    for series_dir in discover_tiff_series(data_dir, channel_labels=CHANNEL_LABELS):
        rows.append({
            "lif_name": series_dir.parent.name,
            "series_name": series_dir.name,
            "series_dir": series_dir,
            "stacks_dir": series_dir / "stacks",
            "denoised_stacks_dir": series_dir / "denoised_stacks",
            "metadata_path": series_dir / "metadata.json",
        })
    return pd.DataFrame(rows)

series_df = discover_series()
display(series_df)

## 3. Save and QC Helpers

In [ ]:
def _pixel_sizes_from_metadata(metadata: dict) -> tuple[float | None, float | None]:
    scale = metadata.get("scale") or {}
    try:
        pixel_size_um_xy = float(scale["x"]) if scale.get("x") is not None else None
    except (TypeError, ValueError):
        pixel_size_um_xy = None
    try:
        pixel_size_um_z = float(scale["z"]) if scale.get("z") is not None else None
    except (TypeError, ValueError):
        pixel_size_um_z = None
    return pixel_size_um_xy, pixel_size_um_z


def stack_channels(stacks: dict[str, np.ndarray], channel_labels: tuple[str, ...] = CHANNEL_LABELS) -> np.ndarray:
    return np.stack([ensure_uint16(stacks[label]) for label in channel_labels], axis=1)


def save_denoised_series(series: SeriesData, diagnostics: dict) -> dict:
    series_dir = DATA_DIR / sanitize_name(series.lif_name) / sanitize_name(series.series_name)
    denoised_dir = series_dir / "denoised_stacks"
    denoised_dir.mkdir(parents=True, exist_ok=True)

    pixel_size_um_xy, pixel_size_um_z = _pixel_sizes_from_metadata(series.metadata)
    stack_path = denoised_dir / f"{sanitize_name(series.series_name)}_stack_denoised.tif"
    _write_imagej_tiff(
        path=stack_path,
        arr_zcyx=stack_channels(series.stacks),
        pixel_size_um_xy=pixel_size_um_xy,
        pixel_size_um_z=pixel_size_um_z,
        channel_labels=CHANNEL_LABELS,
        extra_json={
            "lif_name": series.lif_name,
            "series_name": series.series_name,
            "channel_labels": list(CHANNEL_LABELS),
            "preprocessing": {
                "method": "scanner-line denoise from reference/lif_pipeline.py",
                "denoise_detection_neighborhood": DENOISE_DETECTION_NEIGHBORHOOD,
                "denoise_detection_threshold": DENOISE_DETECTION_THRESHOLD,
                "denoise_min_punctum_size": DENOISE_MIN_PUNCTUM_SIZE,
                "denoise_diagnostics_summary": _summarize_denoise_diag(diagnostics),
            },
        },
    )

    metadata = dict(series.metadata)
    metadata["preprocessing"] = {
        "method": "scanner-line denoise from reference/lif_pipeline.py",
        "denoise_lines": True,
        "denoise_detection_neighborhood": DENOISE_DETECTION_NEIGHBORHOOD,
        "denoise_detection_threshold": DENOISE_DETECTION_THRESHOLD,
        "denoise_min_punctum_size": DENOISE_MIN_PUNCTUM_SIZE,
        "denoise_diagnostics_summary": _summarize_denoise_diag(diagnostics),
        "denoised_stack_path": str(stack_path),
    }
    metadata_path = series_dir / "preprocessing_metadata.json"
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)

    return {"denoised_stack_path": stack_path, "preprocessing_metadata_path": metadata_path}


def save_denoise_qc(series: SeriesData, diagnostics: dict) -> Path | None:
    candidates = []
    for label, diag in diagnostics.items():
        candidates.append((diag.get("total_pixels_zeroed", 0), label, diag))
    if not candidates:
        return None
    _, label, diag = max(candidates, key=lambda item: item[0])
    worst_z = int(diag.get("worst_z", 0))
    raw = diag["worst_raw"].astype(np.float32)
    cleaned = diag["worst_cleaned"].astype(np.float32)
    diff = cleaned - raw

    series_dir = DATA_DIR / sanitize_name(series.lif_name) / sanitize_name(series.series_name)
    qc_dir = series_dir / "denoise_qc"
    qc_dir.mkdir(parents=True, exist_ok=True)
    out_path = qc_dir / f"{sanitize_name(series.series_name)}_denoise_validation.png"

    vmin, vmax = np.percentile(raw, 1), np.percentile(raw, 99.5)
    dmax = max(float(np.percentile(np.abs(diff), 99)), 1.0)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(
        f"Line denoising validation - {series.series_name} | {label} | Z={worst_z}\n"
        f"neighborhood={DENOISE_DETECTION_NEIGHBORHOOD}, threshold={DENOISE_DETECTION_THRESHOLD} MAD, "
        f"min_punctum_size={DENOISE_MIN_PUNCTUM_SIZE}",
        fontsize=12,
    )
    axes[0].imshow(raw, cmap="gray", vmin=vmin, vmax=vmax)
    axes[0].set_title("Original")
    axes[1].imshow(cleaned, cmap="gray", vmin=vmin, vmax=vmax)
    axes[1].set_title("Cleaned")
    axes[2].imshow(diff, cmap="RdBu_r", vmin=-dmax, vmax=dmax)
    axes[2].set_title("Difference")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    return out_path

## 4. Run Denoise

In [ ]:
denoise_results = []

if not RUN_DENOISE:
    print("[SKIP] RUN_DENOISE=False")
elif series_df.empty:
    print(f"[SKIP] No stack series found under {DATA_DIR}")
else:
    for row in series_df.itertuples(index=False):
        series = load_series_from_tiff_dir(Path(row.series_dir), channel_labels=CHANNEL_LABELS)
        if series is None:
            print(f"[SKIP] Could not load {row.series_dir}")
            continue

        print(f"\n[DENOISE] {series.lif_name} :: {series.series_name}")
        diagnostics = denoise_scanner_lines_series(
            series=series,
            detection_neighborhood=DENOISE_DETECTION_NEIGHBORHOOD,
            detection_threshold=DENOISE_DETECTION_THRESHOLD,
            min_punctum_size=DENOISE_MIN_PUNCTUM_SIZE,
            channel_labels=CHANNEL_LABELS,
            verbose=True,
        )

        saved = {}
        if SAVE_DENOISED_STACKS:
            saved = save_denoised_series(series, diagnostics)
        qc_path = save_denoise_qc(series, diagnostics) if SAVE_QC_FIGURES else None

        summary = _summarize_denoise_diag(diagnostics)
        denoise_results.append({
            "lif_name": series.lif_name,
            "series_name": series.series_name,
            "denoised_stack_path": str(saved.get("denoised_stack_path", "")),
            "preprocessing_metadata_path": str(saved.get("preprocessing_metadata_path", "")),
            "qc_path": str(qc_path) if qc_path else "",
            "total_pixels_zeroed": sum(v["total_pixels_zeroed"] for v in summary.values()),
            "total_bad_rows": sum(v["total_bad_rows"] for v in summary.values()),
        })

results_df = pd.DataFrame(denoise_results)
display(results_df)

## 5. Output Summary

In [ ]:
if "results_df" in globals() and not results_df.empty:
    print("Denoise outputs written:")
    for result in results_df.itertuples(index=False):
        print(f"- {result.lif_name}/{result.series_name}")
        print(f"  stack: {result.denoised_stack_path}")
        print(f"  metadata: {result.preprocessing_metadata_path}")
        print(f"  qc: {result.qc_path}")
else:
    print("No denoise outputs were generated.")